# Results 6 — Genetic evidence and therapeutic success

Enrichment of GWAS genetic support among approved target-indication pairs, how it varies with
variant frequency, effect size, protein-altering status and gene-level pleiotropy, and the
combined criterion that performs best.

| file | panel |
| --- | --- |
| `temporal_drug_enrichment_full_chembl.csv` | Figure 5a |
| `drug_enrichment_subsets_vs_full_l2g.csv` | Figure 5b |
| `df_for_enrichment_regression.csv` | Figure 5c |

All strata are computed from the one pair-level table `ti_pairs_chembl`, so no definition can
drift between rows. Odds ratios are Fisher's exact; relative success is a risk ratio.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from manuscript_methods import paper
from manuscript_methods.enrichment import bh_plain, contrast, or_rs, support_mask

numbers = {}
master = pd.read_parquet(paper.derived("ti_pairs_chembl"))
approved = master["approved"].to_numpy().astype(bool)
print("target-indication pairs:", len(master), "| approved:", int(approved.sum()))

target-indication pairs: 37377 | approved: 4564


## Overall enrichment

In [2]:
overall = or_rs(support_mask(master), master["approved"])
numbers["R6.01"] = overall["yes_evid-high_clinphase"]
numbers["R6.02"] = round(overall["odds_ratio"], 2)
numbers["R6.03"] = round(overall["relative_success"], 2)
print({k: numbers[k] for k in ["R6.01", "R6.02", "R6.03"]})
print("P(OR) = %.2e  P(RS) = %.2e" % (overall["p_value"], overall["rs_p_value"]))

{'R6.01': 242, 'R6.02': 3.62, 'R6.03': 2.76}
P(OR) = 3.53e-49  P(RS) = 3.16e-77


## Figure 5b — enrichment by class of genetic support

Each group contrasts a restricted definition of support against the complement of that
restriction among supported pairs. Rare means the supporting credible set has MAF below 0.01,
large effect means a rescaled absolute effect above 0.5, PAV means the credible set contains a
protein-altering variant.

In [3]:
# Each group contrasts a restricted definition of genetic support against the rest of the
# supported pairs. Both rows are measured against pairs with no genetic support at all, which
# is the reference the published panel uses.
STRATA = {
    "PAV": ("PAV_subEvid", "PAV_base", "score_pav"),
    "effect size": ("BigEffect_subEvid", "BigEffect_base", "score_large_effect"),
    "variant frequency": ("rare_subEvid", "rare_base", "score_rare"),
}
PLEIOTROPY_STRATA = {
    "gPS": (("low-gPS-5_subEvid", {"gps_min": 1, "gps_max": 5}), ("high-gPS_subEvid", {"gps_min": 10})),
    "therapeutic areas": (("TA-1_subEvid", {"ta_min": 1, "ta_max": 1}), ("TA-6plus_subEvid", {"ta_min": 6})),
}

any_support = support_mask(master)
unsupported = ~any_support


def against_unsupported(mask):
    """Odds ratio of one stratum against the pairs with no genetic support."""
    keep = (mask | unsupported).to_numpy()
    return or_rs(mask[keep], master["approved"][keep])


def group_rows(group, masks):
    """Two forest rows and the contrast between them."""
    (first_label, first), (second_label, second) = masks
    rows = [
        {"group": group, "datasource": label, **against_unsupported(mask)}
        for label, mask in [(first_label, first), (second_label, second)]
    ]
    counts = {
        "x_low": int((first & approved).sum()),
        "n_low": int((first & ~approved).sum()),
        "x_high": int((second & approved).sum()),
        "n_high": int((second & ~approved).sum()),
    }
    return rows, {"group": group, "low": first_label, "high": second_label, **counts, **contrast(**counts)}


forest = [{"group": "All GWAS", "datasource": "full_l2g", **overall}]
tests = []
for group, (subset_label, base_label, column) in STRATA.items():
    subset = support_mask(master, score_column=column)
    rows, test = group_rows(group, [(subset_label, subset), (base_label, any_support & ~subset)])
    forest += rows
    tests.append(test)
# The two pleiotropy groups were published with different reference groups: gPS against the
# unsupported pairs, therapeutic areas against every other pair. Both are kept as published.
for group, ((low_label, low_kwargs), (high_label, high_kwargs)) in PLEIOTROPY_STRATA.items():
    masks = [(low_label, support_mask(master, **low_kwargs)), (high_label, support_mask(master, **high_kwargs))]
    if group == "therapeutic areas":
        rows = [{"group": group, "datasource": label, **or_rs(mask, master["approved"])} for label, mask in masks]
        counts = {
            "x_low": int((masks[0][1] & approved).sum()),
            "n_low": int((masks[0][1] & ~approved).sum()),
            "x_high": int((masks[1][1] & approved).sum()),
            "n_high": int((masks[1][1] & ~approved).sum()),
        }
        test = {"group": group, "low": masks[0][0], "high": masks[1][0], **counts, **contrast(**counts)}
    else:
        rows, test = group_rows(group, masks)
    forest += rows
    tests.append(test)

forest = pd.DataFrame(forest)
tests = pd.DataFrame(tests)
tests["fdr"] = bh_plain(tests["p_value"])

# Column layout figure_5.R reads.
forest = forest.rename(columns={"relative_success": "Relative success", "n_support": "total_indirect_assoc"})
forest["drugsource"] = "full_chembl"
forest["diffence_pval"] = forest["group"].map(dict(zip(tests["group"], tests["p_value"])))
forest.to_csv(paper.derived("drug_enrichment_subsets_vs_full_l2g.csv"), index=False)
tests.to_csv(paper.derived("figure_5b_contrasts.csv"), index=False)

print(
    forest[["group", "datasource", "odds_ratio", "ci_low", "ci_high", "yes_evid-high_clinphase"]]
    .round(4)
    .to_string(index=False)
)
print()
print(tests[["group", "low", "high", "p_value", "fdr"]].round(5).to_string(index=False))

            group        datasource  odds_ratio  ci_low  ci_high  yes_evid-high_clinphase
         All GWAS          full_l2g      3.6186  3.0936   4.2326                      242
              PAV       PAV_subEvid      6.0483  4.4260   8.2654                       72
              PAV          PAV_base      3.0924  2.5791   3.7080                      170
      effect size BigEffect_subEvid      4.6282  3.1005   6.9087                       39
      effect size    BigEffect_base      3.4730  2.9316   4.1144                      203
variant frequency      rare_subEvid      6.9941  4.2111  11.6163                       29
variant frequency         rare_base      3.3955  2.8789   4.0047                      213
              gPS low-gPS-5_subEvid      4.7983  3.6532   6.3024                       86
              gPS  high-gPS_subEvid      2.9677  2.3595   3.7328                      104
therapeutic areas      TA-1_subEvid      4.2907  2.5291   7.2794                       22
therapeuti

In [4]:
by_stratum = forest.set_index("datasource")["odds_ratio"]
by_group = tests.set_index("group")

numbers["R6.05"] = float(by_stratum["rare_subEvid"])
numbers["R6.06"] = float(by_stratum["rare_base"])
numbers["R6.07"] = round(float(by_group.loc["variant frequency", "p_value"]), 4)
numbers["R6.13"] = float(by_stratum["PAV_subEvid"])
numbers["R6.14"] = float(by_stratum["PAV_base"])
numbers["R6.15"] = round(float(by_group.loc["PAV", "p_value"]), 4)
numbers["R6.17"] = float(by_stratum["BigEffect_subEvid"])
numbers["R6.18"] = float(by_stratum["BigEffect_base"])
numbers["R6.19"] = float(by_stratum["low-gPS-5_subEvid"])
numbers["R6.20"] = float(by_stratum["high-gPS_subEvid"])
numbers["R6.21"] = round(float(by_group.loc["gPS", "p_value"]), 3)
numbers["R6.22"] = float(by_stratum["TA-1_subEvid"])
numbers["R6.23"] = float(by_stratum["TA-6plus_subEvid"])
numbers["R6.24"] = round(float(by_group.loc["therapeutic areas", "p_value"]), 2)
print({k: numbers[k] for k in sorted(numbers) if k >= "R6.05"})

{'R6.05': 6.994051439745637, 'R6.06': 3.3954651611381843, 'R6.07': 0.0077, 'R6.13': 6.048323445762209, 'R6.14': 3.092428147282449, 'R6.15': 0.0002, 'R6.17': 4.6282475044622196, 'R6.18': 3.4730186783176276, 'R6.19': 4.798286448368984, 'R6.20': 2.9677312242353167, 'R6.21': 0.008, 'R6.22': 4.2907160793554455, 'R6.23': 2.8881755914500533, 'R6.24': 0.18}


## Rare disease resources and gene-based tests

Each resource is propagated through the ontology and tested against the same ChEMBL pairs, using
the library implementation the published analysis used.

In [5]:
from gentropy.common.session import Session
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

disease_index = session.spark.read.parquet(paper.release("disease") + "/disease.parquet")
chembl_evidence = session.spark.read.parquet(paper.release("evidence") + "/sourceId=chembl")
all_evidence = session.spark.read.parquet(paper.release("evidence"))
combined = session.spark.read.parquet(paper.baseline("combined_evidence_with_measurements"))
print(combined.groupBy("source").count().toPandas().to_string(index=False))

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/19 01:51:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


           source  count
     all_diseases  36858
      gene_burden   6857
             omim   6596
         orphanet   6192
 all_measurements 150360
           ChEMBL  25234
    cancer_ChEMBL   8811
        gwas_eQTL  12343
    gwas_with_pav   5441
           molQTL  17755
non_cancer_ChEMBL  16423


In [6]:
def enrichment_of(evidence, label, threshold=0.0):
    """Odds ratio and relative success for one evidence source against the ChEMBL pairs."""
    table = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
        evid=evidence,
        disease_index_orig=disease_index,
        chembl_orig=chembl_evidence,
        indirect_assoc_score_thr=threshold,
        efo_ancestors_to_remove=["MONDO_0045024"],
    )
    table["datasource"] = label
    return table


def platform_evidence(datasources, score=0.75):
    """Release evidence rows from the given datasources above a score threshold."""
    return (
        all_evidence.filter(f.col("score") >= score)
        .filter(f.col("datasourceId").isin(datasources))
        .drop("resourceScore")
        .withColumn("resourceScore", f.lit(1.0))
    )


sources = [
    ("OMIM", combined.filter(f.col("source") == "omim")),
    ("Orphanet", combined.filter(f.col("source") == "orphanet")),
    ("Gene-based tests", combined.filter(f.col("source") == "gene_burden")),
    ("ClinVar/ClinGen", platform_evidence(["eva", "clingen"])),
    ("UniProt", platform_evidence(["uniprot_variants", "uniprot_literature"])),
    ("The Genomics England PanelApp", platform_evidence(["genomics_england"])),
]

resources = pd.concat([enrichment_of(evidence, label) for label, evidence in sources], ignore_index=True)
resources.to_csv(paper.derived("drug_enrichment_other_resources.csv"), index=False)
approved_rows = resources[resources["clinicalPhase"] == "4+"].set_index("datasource")
print(approved_rows[["odds_ratio", "Relative success", "yes_evid-high_clinphase"]].round(3).to_string())

                               odds_ratio  Relative success  yes_evid-high_clinphase
datasource                                                                          
OMIM                                5.344             3.519                      142
Orphanet                            5.007             3.384                      126
Gene-based tests                    7.218             4.109                       21
ClinVar/ClinGen                     5.074             3.426                      212
UniProt                             5.008             3.385                      134
The Genomics England PanelApp       4.517             3.181                      147


In [7]:
for key, label in [
    ("R6.08", "Orphanet"),
    ("R6.09", "OMIM"),
    ("R6.10", "ClinVar/ClinGen"),
    ("R6.11", "UniProt"),
    ("R6.12", "The Genomics England PanelApp"),
    ("R6.16", "Gene-based tests"),
]:
    numbers[key] = round(float(approved_rows.loc[label, "odds_ratio"]), 1)
print({k: numbers[k] for k in ["R6.08", "R6.09", "R6.10", "R6.11", "R6.12", "R6.16"]})

{'R6.08': 5.0, 'R6.09': 5.3, 'R6.10': 5.1, 'R6.11': 5.0, 'R6.12': 4.5, 'R6.16': 7.2}


## Figure 5c — probability of success against pleiotropy

The regression frame is the pair-level table with the propagated variant features. Non-linearity
is tested by likelihood ratio against a linear pleiotropy term.

In [8]:
regression = master[
    [
        "targetId",
        "diseaseId",
        "score_all",
        "max_beta",
        "min_maf",
        "max_vep",
        "maxClinicalPhase",
        "uniqueDiseases",
        "uniqueTherapeuticAreas",
    ]
].copy()
regression = regression.rename(columns={"score_all": "indirect_assoc_score"}).fillna(
    {
        "indirect_assoc_score": 0.0,
        "max_beta": 0.0,
        "min_maf": 0.0,
        "max_vep": 0.0,
        "uniqueDiseases": 0.0,
        "uniqueTherapeuticAreas": 0.0,
    }
)
regression["outcome"] = (regression["maxClinicalPhase"] >= 4).astype(int)
regression["geneticSupport"] = (regression["indirect_assoc_score"] >= 0.1).astype(int)
regression.to_csv(paper.derived("df_for_enrichment_regression.csv"), index=False)
print(regression.shape)
regression.head(3)

(37377, 11)


,targetId,diseaseId,indirect_assoc_score,max_beta,min_maf,max_vep,maxClinicalPhase,uniqueDiseases,uniqueTherapeuticAreas,outcome,geneticSupport
0,ENSG00000007314,EFO_0000555,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0,0
1,ENSG00000007314,EFO_0004699,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0,0
2,ENSG00000007314,EFO_0801084,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0,0


In [9]:
def nonlinearity(pleiotropy):
    """Likelihood ratio test of a log pleiotropy term against a linear one."""
    df = regression.copy()
    df["pleiotropy"] = df[pleiotropy]
    df["log_pleiotropy"] = np.log1p(df["pleiotropy"])
    baseline = smf.logit("outcome ~ geneticSupport", data=df).fit(disp=False)
    linear = smf.logit("outcome ~ geneticSupport + pleiotropy", data=df).fit(disp=False)
    log_model = smf.logit("outcome ~ geneticSupport + pleiotropy + log_pleiotropy", data=df).fit(disp=False)
    return {
        "pleiotropy": pleiotropy,
        "llr_vs_baseline": float(log_model.llr_pvalue),
        "p_log_term": float(log_model.pvalues["log_pleiotropy"]),
        "p_linear_term": float(linear.pvalues["pleiotropy"]),
        "llf_baseline": float(baseline.llf),
        "llf_linear": float(linear.llf),
        "llf_log": float(log_model.llf),
    }


tests_nonlinear = pd.DataFrame([nonlinearity("uniqueDiseases"), nonlinearity("uniqueTherapeuticAreas")])
tests_nonlinear.to_csv(paper.derived("figure_5c_nonlinearity.csv"), index=False)
tests_nonlinear

,pleiotropy,llr_vs_baseline,p_log_term,p_linear_term,llf_baseline,llf_linear,llf_log
0,uniqueDiseases,3.845530e-57,9.669350e-12,0.640442,-13762.253839,-13762.143946,-13738.229188
1,uniqueTherapeuticAreas,7.956223e-64,3.958654e-18,0.588899,-13762.253839,-13762.108492,-13722.783381


## High pleiotropy against no genetic support

Highly pleiotropic supported pairs are still more successful than clinical candidates with no
GWAS support for that pair, which is the reference the odds ratio below is taken against.

In [10]:
high_pleiotropy = support_mask(master, gps_min=10)
unsupported = ~support_mask(master)
subset = pd.DataFrame({"outcome": master["approved"], "support": np.where(high_pleiotropy, 1, 0)})[
    (high_pleiotropy | unsupported).to_numpy()
]
model = smf.logit("outcome ~ support", data=subset).fit(disp=False)
print("OR high pleiotropy vs no support: %.3f  P = %.2e" % (np.exp(model.params["support"]), model.pvalues["support"]))
numbers["R6.25"] = round(float(np.exp(model.params["support"])), 2)

OR high pleiotropy vs no support: 2.968  P = 1.46e-20


## The combined criterion

Protein-altering support with genetic support in two to five therapeutic areas. The definition
comes from the two observations above, not from a search over thresholds.

In [11]:
strict_mask = support_mask(master, pav=True, ta_min=2, ta_max=5)
strict = or_rs(strict_mask, master["approved"])
numbers["R6.28"] = round(strict["odds_ratio"], 1)
numbers["R6.29"] = round(strict["relative_success"], 1)
numbers["R6.30"] = strict["yes_evid-high_clinphase"]
numbers["R6.31"] = round(100 * strict["yes_evid-high_clinphase"] / numbers["R6.01"], 1)
print({k: numbers[k] for k in ["R6.28", "R6.29", "R6.30", "R6.31"]})
print(
    "2x2:",
    [
        [strict["no_evid-low_clinphase"], strict["no_evid-high_clinphase"]],
        [strict["yes_evid-low_clinphase"], strict["yes_evid-high_clinphase"]],
    ],
)

{'R6.28': 10.3, 'R6.29': 4.8, 'R6.30': 51, 'R6.31': 21.1}
2x2: [[32777, 4513], [36, 51]]


In [12]:
# Supplementary Table 7: the gene-disease associations meeting the criterion.
genes = pd.read_parquet(paper.derived("gene_table"))[["geneId", "uniqueTherapeuticAreas"]]
window = set(genes.loc[genes["uniqueTherapeuticAreas"].between(2, 5), "geneId"])

l2g = pd.read_parquet(paper.derived("prioritised_genes_diseases"))[
    ["geneId", "diseaseIds", "VEP", "variantId", "score"]
]
pav_rows = l2g[(l2g["VEP"] == 1) & l2g["geneId"].isin(window)].explode("diseaseIds")
associations = pav_rows[["geneId", "diseaseIds"]].dropna().drop_duplicates()
numbers["R6.27"] = len(associations)
associations.to_csv(paper.derived("st7_pav_gene_disease_pairs.csv"), index=False)
print("gene-disease associations meeting the criterion:", numbers["R6.27"])

gene-disease associations meeting the criterion: 2734


## Previously approved targets

Whether a target already approved for another indication is more likely to succeed.

In [13]:
supported = master[support_mask(master)].copy()
approvals_per_target = master.loc[master["approved"] == 1, "targetId"].value_counts()
# "Previously approved" means approved for a different indication, so the pair itself is
# discounted; otherwise every approved pair trivially qualifies.
other_approvals = supported["targetId"].map(approvals_per_target).fillna(0) - supported["approved"]
supported["previously_approved"] = (other_approvals > 0).astype(int)
repurposing = or_rs(supported["previously_approved"].astype(bool), supported["approved"])
numbers["R6.26"] = round(repurposing["odds_ratio"], 2)
print("OR for a previously approved target:", numbers["R6.26"])

OR for a previously approved target: 8.71


## Figure 5a — enrichment over time

For each year, only the credible sets published up to that year contribute genetic support, and
the enrichment is recomputed against the full ChEMBL pair set.

In [14]:
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus

sl = StudyLocus.from_parquet(session, paper.release("credible_set"))
si = StudyIndex.from_parquet(session, paper.release("study"))
l2g_spark = session.spark.read.parquet(paper.derived("prioritised_genes_diseases"))

yearly = []
for year in range(2008, 2026):
    evidence = chemblDrugEnrichment.to_disease_target_evidence(
        table_with_score=l2g_spark.filter(f.col("year") <= year).drop("diseaseIds"),
        score_column="score",
        datasource_id="l2g",
        study_locus=sl,
        study_index=si,
        min_score=0.1,
    )
    table = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
        evid=evidence,
        disease_index_orig=disease_index,
        chembl_orig=chembl_evidence,
        indirect_assoc_score_thr=0.1,
        efo_ancestors_to_remove=["MONDO_0045024"],
    )
    table["datasource"] = str(year)
    table["drugsource"] = "full_chembl"
    yearly.append(table)
    print(year, "done")

temporal = pd.concat(yearly, ignore_index=True)
temporal.to_csv(paper.derived("temporal_drug_enrichment_full_chembl.csv"), index=False)
temporal[temporal["clinicalPhase"] == "4+"][["datasource", "odds_ratio", "yes_evid-high_clinphase"]].round(3)

2008 done


2009 done


2010 done


2011 done


2012 done


...

2022 done


2023 done


2024 done


2025 done


,datasource,odds_ratio,yes_evid-high_clinphase
2,2008,12.599,7
5,2009,15.454,15
8,2010,6.183,18
11,2011,4.879,25
14,2012,4.603,35
17,2013,4.608,42
20,2014,4.542,47
23,2015,3.811,51
26,2016,3.760,57
29,2017,3.717,67


## Numbers

In [15]:
print(paper.save_results("therapeutic_success", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/therapeutic_success.json


,computed
R6.01,242.000000
R6.02,3.620000
R6.03,2.760000
R6.05,6.994051
R6.06,3.395465
R6.07,0.007700
R6.13,6.048323
R6.14,3.092428
R6.15,0.000200
R6.17,4.628248
